# **Multithreading**

In [1]:
import threading
import time

def sleeper_fn(n:int):
    print(f"Sleeping for {n} seconds")
    time.sleep(n)
    print(f"\nFinished Sleeping for {n} seconds")

#creating threads

t1 = threading.Thread(target= sleeper_fn, args=(1,))
#                                           ^
#                                           |
#                                           |
# if using tuple need to put a comma at the end for single args
# but if there are 2 args dont need the comma at the end eg. below
# t1 = threading.Thread(target= sleeper_fn, args=(1,2))
t2 = threading.Thread(target= sleeper_fn, args=(2,))
t3 = threading.Thread(target= sleeper_fn, args=(4,))

# using list at args
# t1 = threading.Thread(target= sleeper_fn, args=[1])
# t2 = threading.Thread(target= sleeper_fn, args=[2])
# t3 = threading.Thread(target= sleeper_fn, args=[4])

# start threads
start = time.time()
t1.start()
t2.start()
t3.start()
#the threads have started but the work hasnt finished

# Wait for completion
t1.join()
t2.join()
t3.join()

print(f"\nFinished execution in {time.time() - start:.2f}")


Sleeping for 1 seconds
Sleeping for 2 seconds
Sleeping for 4 seconds

Finished Sleeping for 1 seconds

Finished Sleeping for 2 seconds

Finished Sleeping for 4 seconds

Finished execution in 4.01


## ThreadPoolExecutor

In [2]:
from concurrent.futures import ThreadPoolExecutor

def worker(task):
    print(f"Task {task} running")
    return(f"Task {task} Completed")


with ThreadPoolExecutor() as executor:
    # Submit two tasks to run in parallel
    result1 = executor.submit(worker, 1)
    result2 = executor.submit(worker, 4)

    # result1 is a future object. its like saying
    #  "The task is running (or will run). I'll hold the result later."
# Create Future object(result 1) -> Run task in thread ->  Store return value inside result1

    print("\n",type(result1))
    print(result1,'\n')# <Future at 0x78171b6b8e30 state=finished returned str>
    '''
so we need to do result1.result
use .result() only when the function is returning some value
result() is like saying "Wait if needed, then give me the actual returned value."
    '''
    print(result1.result())
    print(result2.result())

Task 1 running

 <class 'concurrent.futures._base.Future'>
<Future at 0x7c593fd556d0 state=finished returned str> 

Task 1 Completed
Task 4 running
Task 4 Completed


future = executor.submit(worker, 2)

**Internally:**

Create Future object ->
 Run task in thread ->
 Store return value inside Future

above code is good but has one problem need to give the arguments manually. to solve this we use map

In [3]:
l = [3,4,5]

with ThreadPoolExecutor() as executor:
        future = executor.submit(worker,2)
        all_result = executor.map(worker,l)

        print('\n',future.result(),'\n')
        ''' notice i need .result() in above line but not in the line below
its bcz map returns an iterator of final results
        '''
        for output in all_result:
            print(output)

        print("\n",type(future))
        print(type(all_result)) # class generator so values are yielded

Task 2 running
Task 3 running
Task 4 running

 Task 2 Completed 

Task 3 Completed
Task 4 Completed
Task 5 running
Task 5 Completed

 <class 'concurrent.futures._base.Future'>
<class 'generator'>


map internally does sth like:

In [4]:
# futures = [executor.submit(worker, x) for x in l]

# for f in futures:
#     yield f.result()

# **Multiprocessing**

In [5]:
# importing the multiprocessing module
import multiprocessing

def print_cube(num):
   print("Cube: {}".format(num * num * num))

def print_square(num):
    print("Square: {}".format(num * num))

if __name__ == "__main__":
    # creating processes
    p1 = multiprocessing.Process(target=print_square, args=(10, ))
    p2 = multiprocessing.Process(target=print_cube, args=(10, ))

    # starting processes
    p1.start()
    p2.start()

    # wait until processes are finished
    p1.join()
    p2.join()

    # both processes finished
    print("Done!")

Square: 100
Cube: 1000
Done!


Array: allocates array to sahre multiple values of same type

Value: shares a single value


## Lock

In [6]:
from multiprocessing import Process, Lock, Value

def worker(lock, counter):
    for _ in range(10000):
        # with lock:  # trying without lock, we get different value
            counter.value += 1

if __name__ == "__main__":
    lock = Lock()
    counter = Value('i', 0)   # shared, synchronized integer

    list_of_processes = [Process(target=worker, args=(lock, counter)) for _ in range(4)]

    for p in list_of_processes:
        p.start()
    for p in list_of_processes:
        p.join()

    print(counter.value)


27905


## Semaphore

Using lock only one process can enter the critical section at a time. But by using semaphore we can specify how many workers can enter the critical section at a time.


In [7]:
import time
import random
from multiprocessing import Process, Semaphore

def worker(sem, idx):
    print(f"Worker {idx} waiting for semaphore…")

    with sem:
        print(f"→ Worker {idx} ENTERED critical section ({sem})")
        # simulate some work
        time.sleep(random.uniform(1, 3))
        print(f"← Worker {idx} LEAVING critical section ({sem})")
    print(f"← Worker {idx} LEFT critical section ({sem})")

if __name__ == "__main__":
    # only 2 workers may hold the semaphore at once
    sem = Semaphore(2)

    procs = [Process(target=worker, args=(sem, i)) for i in range(6)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()


Worker 0 waiting for semaphore…
Worker 1 waiting for semaphore…→ Worker 0 ENTERED critical section (<Semaphore(value=1)>)
Worker 2 waiting for semaphore…

→ Worker 1 ENTERED critical section (<Semaphore(value=0)>)
Worker 3 waiting for semaphore…Worker 4 waiting for semaphore…
Worker 5 waiting for semaphore…

← Worker 0 LEAVING critical section (<Semaphore(value=0)>)
← Worker 0 LEFT critical section (<Semaphore(value=1)>)→ Worker 2 ENTERED critical section (<Semaphore(value=0)>)

← Worker 1 LEAVING critical section (<Semaphore(value=0)>)
→ Worker 3 ENTERED critical section (<Semaphore(value=0)>)← Worker 1 LEFT critical section (<Semaphore(value=0)>)

← Worker 2 LEAVING critical section (<Semaphore(value=0)>)
← Worker 2 LEFT critical section (<Semaphore(value=1)>)→ Worker 4 ENTERED critical section (<Semaphore(value=0)>)

← Worker 3 LEAVING critical section (<Semaphore(value=0)>)
← Worker 3 LEFT critical section (<Semaphore(value=1)>)→ Worker 5 ENTERED critical section (<Semaphore(value=

# Simulating Race Around Condition

## multithreading

In [8]:
import threading
import time

counter = 0

def increase_or_decrease_counter(a):
    global counter
    for _ in range(1000):
        local_counter = counter

        time.sleep(0)#releases GIL

        counter = local_counter + a

threads = []
num_threads = 5

for _ in range(num_threads):

    t1 = threading.Thread(target=increase_or_decrease_counter,args=(1,))
    threads.append(t1)
    t1.start()

    t2 = threading.Thread(target=increase_or_decrease_counter,args=(-1,))
    threads.append(t2)
    t2.start()

# Wait for all threads to complete
for t in threads:
    t.join()

print(f"Final counter value: {counter}")
print(f"Expected value: {0}")
# print(f"Expected value: {1000 * num_threads}")

Final counter value: -982
Expected value: 0


## multiprocessing

In [9]:
# race around condition doesnt work for multiprocessing bcz each process takes their own rscs
import multiprocessing
import time

counter = 0

def increase_or_decrease_counter(a):
    global counter
    for _ in range(1000):
        local_counter = counter

        time.sleep(0)#releases GIL

        counter = local_counter + a

processes = []
num_processes = 5

for _ in range(num_processes):

    t1 = multiprocessing.Process(target=increase_or_decrease_counter,args=(1,))
    processes.append(t1)
    t1.start()

    # t2 = multiprocessing.Process(target=increase_or_decrease_counter,args=(-1,))
    # processes.append(t2)
    # t2.start()

# Wait for all processes to complete
for t in processes:
    t.join()

print(f"Final counter value: {counter}")
# print(f"Expected value: {0}")
print(f"Expected value: {1000 * num_processes}")

Final counter value: 0
Expected value: 5000
